# Paper 2 report figures
This notebook generates the report's three figures exclusively from the saved final-evidence tables. It validates source coverage before plotting and does not train models or change the evidence packages.


## 1. Load immutable evidence
The setup locates the repository from the notebook environment, imports the tracked extractor, and checks all source counts and routing audits before plotting.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo = next(parent for parent in (cwd, *cwd.parents) if (parent / "data/splits").is_dir() and (parent / "notebooks").is_dir())
# Own assembly root: the nearest directory above the working directory that
# carries this notebook plus its scripts and templates. Never hardcode another
# assembly version here; run this notebook from its own directory
# (writeup2/report-assemble-X.Y/notebooks) or from the assembly root itself.
candidates = [
    parent for parent in (cwd, *cwd.parents)
    if (parent / "scripts" / "report_data.py").is_file()
    and (parent / "templates" / "report.md").is_file()
    and (parent / "notebooks" / "report_figures.ipynb").is_file()
]
if not candidates:
    raise RuntimeError(
        "report assembly root not found above working directory "
        f"{cwd}; execute this notebook from its own assembly directory "
        "(writeup2/report-assemble-X.Y/notebooks) or the assembly root."
    )
report_root = candidates[0]
sys.path.insert(0, str(report_root / "scripts"))
import report_data as rd
rd.validate_sources()
figures = report_root / "figures"
figures.mkdir(parents=True, exist_ok=True)
print(f"Source validation passed; figures will be written to {figures}")


Source validation passed; figures will be written to /scratch/group/p.cis250607.000/MDR-Project/writeup2/report-assemble-1.5/figures


## 2. Temporal and paired LOSO comparison
The left panel shows the temporal rows by evidence role: the three seed-paired flagship rows and the four labeled heuristic context rows. The right panel plots per-station paired LOSO differences for both flagship pairings: station-majority versus the model without the station consistency guarantee, and station-majority versus the paired global refit.

In [2]:
import statistics
main_seed = pd.DataFrame(rd.read_csv("main_seed"))
global_temporal = pd.DataFrame(rd.read_csv("global_temporal_seed"))
loso = pd.DataFrame(rd.read_csv("loso_seed"))
global_loso = pd.DataFrame(rd.read_csv("global_loso_seed"))
T29 = 2.0452296428670278  # two-sided 97.5% t quantile, 29 df (same construction as the saved 1.0 intervals)

temporal = []
for strategy, frame, evidence in [
    ("Guarded_Backbone54_k2", main_seed[(main_seed["strategy_name"] == "Guarded_Backbone54_k2") & (main_seed["evidence_role"] == "paired_in_harness")], "paired_in_harness"),
    ("Clustering_Backbone54_k2", main_seed[(main_seed["strategy_name"] == "Clustering_Backbone54_k2") & (main_seed["evidence_role"] == "paired_in_harness")], "paired_in_harness"),
    ("Global_Single", global_temporal, "paired_in_harness"),
    ("Clustering_Dynamic_k2", main_seed[(main_seed["strategy_name"] == "Clustering_Dynamic_k2") & (main_seed["evidence_role"] == "sensitivity_anchored")], "sensitivity_anchored"),
    ("Seasonal_Binary_k2", main_seed[(main_seed["strategy_name"] == "Seasonal_Binary_k2") & (main_seed["evidence_role"] == "sensitivity_anchored")], "sensitivity_anchored"),
    ("Univariate_G_API_k2", main_seed[(main_seed["strategy_name"] == "Univariate_G_API_k2") & (main_seed["evidence_role"] == "sensitivity_anchored")], "sensitivity_anchored"),
    ("Trained_Gating_k2", main_seed[(main_seed["strategy_name"] == "Trained_Gating_k2") & (main_seed["evidence_role"] == "sensitivity_anchored")], "sensitivity_anchored"),
]:
    values = pd.to_numeric(frame["r2"])
    assert len(values) == 30 and frame["seed"].nunique() == 30, strategy
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    half = T29 * sd / (len(values) ** 0.5)
    temporal.append({"strategy": strategy, "evidence": evidence, "mean": mean, "sd": sd, "lo": mean - half, "hi": mean + half})
summary = pd.DataFrame(temporal).sort_values("mean").reset_index(drop=True)
guard_seeds = set(main_seed[(main_seed["strategy_name"] == "Guarded_Backbone54_k2") & (main_seed["evidence_role"] == "paired_in_harness")]["seed"])
assert set(global_temporal["seed"]) == guard_seeds, "temporal seed sets do not match for pairing"
paired_temporal = (
    main_seed[(main_seed["strategy_name"] == "Guarded_Backbone54_k2") & (main_seed["evidence_role"] == "paired_in_harness")][["seed", "r2"]].rename(columns={"r2": "guard"})
    .merge(global_temporal[["seed", "r2"]].rename(columns={"r2": "glob"}), on="seed")
)
paired_temporal["guard"] = pd.to_numeric(paired_temporal["guard"])
paired_temporal["glob"] = pd.to_numeric(paired_temporal["glob"])
temporal_diff = paired_temporal["guard"] - paired_temporal["glob"]

guard_loso = loso[loso["strategy_name"] == "Guarded_Backbone54_k2"][["seed", "station", "r2"]].copy()
back_loso = loso[loso["strategy_name"] == "Clustering_Backbone54_k2"][["seed", "station", "r2"]].copy()
glob_loso = global_loso[["seed", "station", "r2"]].copy()
assert len(guard_loso) == 35 and len(glob_loso) == 35, "LOSO coverage changed"
merged = guard_loso.merge(back_loso, on=["seed", "station"], suffixes=("_guard", "_back")).merge(
    glob_loso.rename(columns={"r2": "r2_global"}), on=["seed", "station"])
for column in ["r2_guard", "r2_back", "r2_global"]:
    merged[column] = pd.to_numeric(merged[column])
merged["d_back"] = merged["r2_guard"] - merged["r2_back"]
merged["d_global"] = merged["r2_guard"] - merged["r2_global"]
stations = merged.groupby("station")[["d_back", "d_global"]].mean().mean(axis=1).sort_values().index.tolist()
short_station = {s: s.replace("_WA_985", "").replace("_WA_990", "").replace("_WA", "") for s in stations}

short = {
    "Guarded_Backbone54_k2": "Station-majority shared-feature\ncluster-routed\nmulti-regime model",
    "Clustering_Backbone54_k2": "Shared-feature cluster-routed\nmulti-regime model\nwithout station consistency guarantee",
    "Global_Single": "Existing single-regime\nglobal model (paired refit)",
    "Clustering_Dynamic_k2": "Three-feature K-means grouping",
    "Seasonal_Binary_k2": "Seasonal grouping",
    "Univariate_G_API_k2": "Precipitation-index grouping",
    "Trained_Gating_k2": "Target-threshold grouping",
}
fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), gridspec_kw={"width_ratios": [1.08, 1]})
y = np.arange(len(summary))
for role, color in [("paired_in_harness", "#087e8b"), ("sensitivity_anchored", "#8073ac")]:
    group = summary[summary["evidence"] == role]
    index = group.index.to_numpy()
    axes[0].errorbar(group["mean"], index,
                     xerr=[group["mean"] - group["lo"], group["hi"] - group["mean"]],
                     fmt="o", color=color, capsize=3, label=role.replace("_", " "))
axes[0].set_yticks(y, [short[name] for name in summary["strategy"]])
axes[0].set_xlabel("Temporal R², 2023–2025")
axes[0].set_title("A. Temporal development-split result")
axes[0].grid(axis="x", alpha=0.25)
axes[0].legend(loc="lower right", fontsize=8)
positions = np.arange(len(stations))
height = 0.36
d_back_mean = [merged[merged["station"] == s]["d_back"].mean() for s in stations]
d_back_sd = [merged[merged["station"] == s]["d_back"].std(ddof=1) for s in stations]
d_global_mean = [merged[merged["station"] == s]["d_global"].mean() for s in stations]
d_global_sd = [merged[merged["station"] == s]["d_global"].std(ddof=1) for s in stations]
axes[1].barh(positions - height / 2, d_back_mean, height=height, xerr=d_back_sd,
             color="#45a7a1", capsize=2, label="vs without-guarantee")
axes[1].barh(positions + height / 2, d_global_mean, height=height, xerr=d_global_sd,
             color="#087e8b", capsize=2, label="vs global")
axes[1].axvline(0, color="#263238", linewidth=0.8)
axes[1].set_yticks(positions, [short_station[s] for s in stations])
axes[1].set_xlabel("Station-majority − comparator mean LOSO R²")
axes[1].set_title("B. Paired held-out-station effects")
axes[1].grid(axis="x", alpha=0.25)
axes[1].legend(fontsize=8)
fig.text(0.01, 0.01, "Flagship rows are seed-paired comparisons under one protocol; heuristic rows are labeled context. 95% intervals and error bars reflect expert-seed variation only.", fontsize=8)
fig.tight_layout(rect=(0, 0.035, 1, 1))
path = figures / "main_temporal_and_loso.png"
fig.savefig(path, dpi=220, bbox_inches="tight")
plt.close(fig)
print(summary[["strategy", "evidence", "mean", "sd"]].to_string(index=False))
print(f"temporal Guarded-Global paired diff: mean={temporal_diff.mean():+.6f}, "
      f"wins={(temporal_diff > 0).sum()}/ties={(temporal_diff == 0).sum()}/losses={(temporal_diff < 0).sum()} over 30 seeds")
table = pd.DataFrame({
    "station": stations,
    "d_back_mean": d_back_mean,
    "d_global_mean": d_global_mean,
})
print(table.to_string(index=False))
print(f"Saved {path}")


                strategy             evidence     mean       sd
       Trained_Gating_k2 sensitivity_anchored 0.735359 0.001095
     Univariate_G_API_k2 sensitivity_anchored 0.767566 0.000938
      Seasonal_Binary_k2 sensitivity_anchored 0.770020 0.001559
           Global_Single    paired_in_harness 0.779794 0.001279
   Clustering_Dynamic_k2 sensitivity_anchored 0.785464 0.000995
Clustering_Backbone54_k2    paired_in_harness 0.811724 0.001369
   Guarded_Backbone54_k2    paired_in_harness 0.811843 0.001369
temporal Guarded-Global paired diff: mean=+0.032049, wins=30/ties=0/losses=0 over 30 seeds
              station  d_back_mean  d_global_mean
           Darrington     0.000000      -0.002094
             Quinault     0.000000       0.010990
              Spokane     0.051448      -0.018871
SourdoughGulch_WA_985     0.059106      -0.019505
    BeaverPass_WA_990     0.000000       0.039646
          Paradise_WA     0.015444       0.041544
        CayusePass_WA     0.000000       0.3552

## 3. ECE policy and site diagnostics
This figure reads only the final-evidence ECE policy and station tables. Hatching identifies one-expert forced diagnostics that are not deployable routing policies.


In [3]:
ece_policy = pd.DataFrame(rd.read_csv("ece_policy"))
ece_station = pd.DataFrame(rd.read_csv("ece_station"))
for column in ["rmse_mean", "rmse_std"]:
    ece_policy[column] = pd.to_numeric(ece_policy[column])
    ece_station[column] = pd.to_numeric(ece_station[column])
family = "Guarded_Backbone54_k2"
guarded = ece_policy[ece_policy["family"] == family].set_index("policy")
global_row = ece_policy[(ece_policy["family"] == "Global_Single_54") & (ece_policy["policy"] == "direct")].iloc[0]
policies = ["as_routed", "auto_hard", "auto_soft", "c0_only", "c1_only", "global direct"]
values = [guarded.loc[policy, "rmse_mean"] for policy in policies[:-1]] + [global_row["rmse_mean"]]
errors = [guarded.loc[policy, "rmse_std"] for policy in policies[:-1]] + [global_row["rmse_std"]]
colors = ["#b45d42", "#087e8b", "#45a7a1", "#9ba3ad", "#b8bec7", "#334e68"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1.1, 1.2]})
y = np.arange(len(policies))
bars = axes[0].barh(y, values, xerr=errors, color=colors, capsize=3)
for position in [3, 4]:
    bars[position].set_hatch("///")
    bars[position].set_edgecolor("#374151")
axes[0].set_yticks(y, ["Static", "Auto hard", "Auto soft", "Forced dry (non-deployable)", "Forced complement (non-deployable)", "Global direct"], fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("Pooled RMSE (m³/m³)")
axes[0].set_title("A. Station-majority ECE policies, five seeds")
axes[0].grid(axis="x", alpha=0.25)
station_names = sorted(ece_station["station"].unique())
short_sites = [name.replace("ECE_", "").replace("Renton_", "R_").replace("BBG_", "B_") for name in station_names]
x = np.arange(len(station_names))
width = 0.26
for offset, fam, policy, label, color in [
    (-width, family, "as_routed", "Station-majority static", "#b45d42"),
    (0, family, "auto_hard", "Station-majority auto hard", "#087e8b"),
    (width, "Global_Single_54", "direct", "Global direct", "#334e68"),
]:
    group = ece_station[(ece_station["family"] == fam) & (ece_station["policy"] == policy)].set_index("station")
    axes[1].bar(x + offset, [group.loc[name, "rmse_mean"] for name in station_names], width,
                yerr=[group.loc[name, "rmse_std"] for name in station_names], color=color, label=label, capsize=2)
axes[1].set_xticks(x, short_sites, rotation=35, ha="right")
axes[1].set_ylabel("Station RMSE (m³/m³)")
axes[1].set_title("B. Five disjoint collaboration sites")
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(fontsize=8)
fig.text(0.01, 0.005, "v1.1 Washington-calibrated availability gate; forced-expert bars are diagnostic only. Late-summer ECE targets are final scoring only.", fontsize=8)
fig.tight_layout(rect=(0, 0.045, 1, 1))
path = figures / "ece_policies_and_stations.png"
fig.savefig(path, dpi=220, bbox_inches="tight")
plt.close(fig)
print(ece_policy[["family", "policy", "deployable", "rmse_mean"]].to_string(index=False))
print(f"Saved {path}")


                  family    policy deployable  rmse_mean
   Clustering_V0_Full_k2 as_routed       True   0.165908
   Clustering_V0_Full_k2 auto_hard       True   0.057768
   Clustering_V0_Full_k2 auto_soft       True   0.057768
   Clustering_V0_Full_k2   c0_only      False   0.057768
   Clustering_V0_Full_k2   c1_only      False   0.192536
Clustering_Backbone54_k2 as_routed       True   0.167431
Clustering_Backbone54_k2 auto_hard       True   0.057768
Clustering_Backbone54_k2 auto_soft       True   0.057768
Clustering_Backbone54_k2   c0_only      False   0.057768
Clustering_Backbone54_k2   c1_only      False   0.192536
   Guarded_Backbone54_k2 as_routed       True   0.167431
   Guarded_Backbone54_k2 auto_hard       True   0.057768
   Guarded_Backbone54_k2 auto_soft       True   0.057768
   Guarded_Backbone54_k2   c0_only      False   0.192536
   Guarded_Backbone54_k2   c1_only      False   0.057768
        Global_Single_54    direct       True   0.058634
Saved /scratch/group/p.cis25060

## 4. Station-majority daily overlay on the ECE sites
The saved final-evidence daily overlay table supplies observations and policy predictions. Forced local-expert curves are labeled as non-deployable diagnostics, separate from the automatic G_API route.


In [4]:
overlay = pd.DataFrame(rd.read_csv("ece_overlay"))
overlay["date"] = pd.to_datetime(overlay["date"])
overlay["y_true"] = pd.to_numeric(overlay["y_true"])
overlay["y_pred_mean"] = pd.to_numeric(overlay["y_pred_mean"])
plot_policies = [
    ("as_routed", "Static route", "#b45d42", "-"),
    ("auto_hard", "Auto hard", "#087e8b", "-"),
    ("c0_only", "Forced dry-assigned (not deployable)", "#9ba3ad", "--"),
    ("c1_only", "Forced complement (not deployable)", "#508eaa", "--"),
]
fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
for ax, (station, group) in zip(axes, overlay.groupby("station_id", sort=True)):
    observed = group[group["policy"] == "as_routed"].sort_values("date")
    ax.plot(observed["date"], observed["y_true"], color="#1f2937", linewidth=1.7, label="Observed daily target")
    for policy, label, color, style in plot_policies:
        series = group[group["policy"] == policy].sort_values("date")
        ax.plot(series["date"], series["y_pred_mean"], color=color, linestyle=style,
                linewidth=1.3 if style == "-" else 1.1, label=label)
    ax.set_ylabel("m³/m³")
    ax.set_title(station.replace("ECE_", ""), loc="left", fontsize=10)
    ax.grid(alpha=0.2)
axes[0].legend(ncol=3, fontsize=8, loc="upper right")
axes[-1].set_xlabel("2026 late-summer evaluation date")
fig.suptitle("Station-majority ECE v3 daily observations and policy outputs", fontsize=14)
fig.text(0.01, 0.005, "All ECE rows use the auxiliary availability route; forced one-expert traces are diagnostic and cannot be deployed as shown.", fontsize=8)
fig.tight_layout(rect=(0, 0.025, 1, 0.975))
path = figures / "ece_guarded_daily_overlay.png"
fig.savefig(path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Overlay rows: {len(overlay)}, sites: {overlay['station_id'].nunique()}, policies: {sorted(overlay['policy'].unique())}")
print(f"Saved {path}")


Overlay rows: 600, sites: 5, policies: ['as_routed', 'auto_hard', 'c0_only', 'c1_only']
Saved /scratch/group/p.cis250607.000/MDR-Project/writeup2/report-assemble-1.5/figures/ece_guarded_daily_overlay.png
